# E2 — Leaky vs clean: quantifying the mAP inflation caused by data leakage
**Headline experiment.** Same YOLO26s, same epochs, trained twice on EACH leaky source dataset:
- **leaky**: the authors' ORIGINAL split (augmentation applied before splitting, so variants of the same source image land in both train and test)
- **clean**: pool all images and re-split 70/20/10 GROUPED BY source image (group-aware, seed 42)

The difference mAP(leaky) − mAP(clean) is the inflation.

**How to run:** GPU T4 x2 · Internet ON · set `ROBOFLOW_API_KEY` (use a NEW key; revoke any old one) · Save & Run All.
No Add Input required. Estimated ~4.5-5 h (QUICK_TEST=True: ~30 min). Checkpoint-resume supported.

In [ ]:
# ==================== CONFIG (E2) ====================
QUICK_TEST = False
EPOCHS   = 2 if QUICK_TEST else 15
FRACTION = 0.15 if QUICK_TEST else 1.0
SEED = 42
ROBOFLOW_API_KEY = ""   # <== PUT YOUR NEW KEY HERE (never make the notebook public while a key is in it)

import os, random, shutil, glob, json, time, yaml
WORK = "/kaggle/working"; os.makedirs(f"{WORK}/results", exist_ok=True)

# RESUME: auto-detect the previous output of THIS notebook (if Added as Input)
STATE_PATH = f"{WORK}/state.json"
for _st in glob.glob("/kaggle/input/**/state.json", recursive=True):
    _s = json.load(open(_st))
    if any(k.endswith("_leaky") or k.endswith("_clean") for k in _s):
        shutil.copy(_st, STATE_PATH); print("E2 CHECKPOINT LOADED FROM:", _st); break
STATE = json.load(open(STATE_PATH)) if os.path.exists(STATE_PATH) else {}
def save_state(): json.dump(STATE, open(STATE_PATH, "w"), indent=1, default=str)
def done(k): return k in STATE
def mark(k, v): STATE[k] = v; save_state()
assert ROBOFLOW_API_KEY, "Set ROBOFLOW_API_KEY (create a NEW key on roboflow.com after revoking the old one)!"
print("[E2] QUICK_TEST=", QUICK_TEST, "| stages done:", list(STATE) or "(none yet)")

In [ ]:
%pip install -q ultralytics roboflow
import torch
assert torch.cuda.is_available(), "Enable GPU T4!"
cap = torch.cuda.get_device_capability(0)
assert cap[0] >= 7, f"GPU sm_{cap[0]}{cap[1]} is too old — select T4 x2!"
print("GPU:", torch.cuda.get_device_name(0), "- OK")

## 1. Download the two source datasets (keeping the authors' leaky split)

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
DATASETS = {
    "B_testcasque": ("testcasque", "ppe-detection-qlq3d", 1),
    "C_mohamed":    ("mohamed-traore-2ekkp", "ppe-detection-l80fg", 5),
}
DS_DIR = {}
for name, (ws, proj, ver) in DATASETS.items():
    out = f"{WORK}/{name}"
    if not os.path.exists(f"{out}/data.yaml"):
        rf.workspace(ws).project(proj).version(ver).download("yolov8", location=out)
    DS_DIR[name] = out
    n = sum(len(glob.glob(f"{out}/{sp}/images/*")) for sp in ["train","valid","test"])
    print(f"{name}: {n} images ->", out)

## 2. Leakage statistics + building the CLEAN variant (re-split by source-image group)

In [ ]:
def src_of(path):
    s = os.path.basename(path)
    return s.split(".rf.")[0] if ".rf." in s else os.path.splitext(s)[0]

def fix_yaml(root):
    y0 = yaml.safe_load(open(f"{root}/data.yaml"))
    names = y0["names"]
    y = {"path": root, "train": "train/images", "val": "valid/images", "test": "test/images",
         "names": names, "nc": len(names)}
    p = f"{root}/data_fixed.yaml"; yaml.safe_dump(y, open(p, "w")); return p

LEAK_STATS = STATE.get("leak_stats", {})
CLEAN_DIR, LEAKY_YAML, CLEAN_YAML = {}, {}, {}
for name, root in DS_DIR.items():
    items = []
    for sp in ["train", "valid", "test"]:
        for img in sorted(glob.glob(f"{root}/{sp}/images/*")):
            lbl = f"{root}/{sp}/labels/{os.path.splitext(os.path.basename(img))[0]}.txt"
            items.append((img, lbl if os.path.exists(lbl) else None, sp))
    groups = {}
    for it in items: groups.setdefault(src_of(it[0]), []).append(it)
    splits_of = {k: {i[2] for i in v} for k, v in groups.items()}
    stats = {"images": len(items), "src_groups": len(groups),
             "groups_in_multi_split": sum(1 for s in splits_of.values() if len(s) > 1),
             "train_val_overlap": sum(1 for s in splits_of.values() if "train" in s and "valid" in s),
             "train_test_overlap": sum(1 for s in splits_of.values() if "train" in s and "test" in s)}
    LEAK_STATS[name] = stats
    print(name, stats)
    # ---- clean split: 70/20/10 by group, deterministic ----
    out_root = f"{WORK}/{name}_clean"
    if not os.path.exists(f"{out_root}/data.yaml"):
        for sp in ["train", "valid", "test"]:
            os.makedirs(f"{out_root}/{sp}/images", exist_ok=True)
            os.makedirs(f"{out_root}/{sp}/labels", exist_ok=True)
        keys = sorted(groups); random.Random(SEED).shuffle(keys)
        total = len(items); acc = 0
        for k in keys:
            sp = "train" if acc < 0.7 * total else ("valid" if acc < 0.9 * total else "test")
            for img, lbl, _ in groups[k]:
                shutil.copy(img, f"{out_root}/{sp}/images/")
                if lbl: shutil.copy(lbl, f"{out_root}/{sp}/labels/")
                acc += 1
        shutil.copy(f"{root}/data.yaml", f"{out_root}/data.yaml")
    CLEAN_DIR[name] = out_root
    LEAKY_YAML[name] = fix_yaml(root); CLEAN_YAML[name] = fix_yaml(out_root)
    for sp in ["train", "valid", "test"]:
        print(f"  clean/{sp}:", len(glob.glob(f"{out_root}/{sp}/images/*")), "images")
mark("leak_stats", LEAK_STATS)

## 3. Train YOLO26s: 4 runs (B-leaky, B-clean, C-leaky, C-clean) — same epochs, same seed

In [ ]:
from ultralytics import YOLO
import pandas as pd
rows = []
for name in DS_DIR:
    for arm, dyaml in [("leaky", LEAKY_YAML[name]), ("clean", CLEAN_YAML[name])]:
        key = f"{name}_{arm}"
        if done(key):
            rows.append(STATE[key]); print("skip:", STATE[key]); continue
        t0 = time.time()
        m = YOLO("yolo26s.pt")
        m.train(data=dyaml, epochs=EPOCHS, imgsz=640, batch=16, seed=SEED, fraction=FRACTION,
                project=f"{WORK}/runs", name=key, exist_ok=True, plots=False, verbose=False)
        v = m.val(data=dyaml, split="val", verbose=False)
        t = m.val(data=dyaml, split="test", verbose=False)
        row = {"dataset": name, "arm": arm, "epochs": EPOCHS,
               "mAP50_val": round(v.box.map50, 4), "mAP50_test": round(t.box.map50, 4),
               "mAP50_95_test": round(t.box.map, 4), "min": round((time.time() - t0) / 60, 1)}
        rows.append(row); mark(key, row); print(row)
        del m; import gc, torch; gc.collect(); torch.cuda.empty_cache()

## 4. Headline result table: inflation caused by leakage

In [ ]:
df = pd.DataFrame(rows)
piv = df.pivot(index="dataset", columns="arm", values="mAP50_test")
piv["inflation_mAP50"] = (piv["leaky"] - piv["clean"]).round(4)
piv95 = df.pivot(index="dataset", columns="arm", values="mAP50_95_test")
piv["inflation_mAP50_95"] = (piv95["leaky"] - piv95["clean"]).round(4)
display(df); display(piv)
df.to_csv(f"{WORK}/results/e2_runs.csv", index=False)
piv.to_csv(f"{WORK}/results/e2_inflation.csv")
json.dump(LEAK_STATS, open(f"{WORK}/results/e2_leak_stats.json", "w"), indent=1)
print("\n>>> E2 CONCLUSION: leakage inflates mAP50 by, on average,",
      round(float(piv['inflation_mAP50'].mean()), 4), "points")

In [ ]:
# Clean up the output: delete the downloaded datasets (keep results/ + the small runs/*/weights)
for name in list(DS_DIR):
    shutil.rmtree(DS_DIR[name], ignore_errors=True)
    shutil.rmtree(f"{WORK}/{name}_clean", ignore_errors=True)
print("Cleaned. Output = results/ + runs/ + state.json")